# Desarrollo G1 - Practica ETL Semana 1

Dominio de negocio: ecommerce/ventas.

Fuente de datos: Kaggle, dataset `olistbr/brazilian-ecommerce`.

Objetivo: preparar un entorno ETL con Docker, PostgreSQL y Python, generar tres DataFrames principales y analizar valores nulos, estadisticos y agrupaciones.

## 1. Configuracion inicial

Se cargan librerias, variables de entorno y rutas de trabajo. La base PostgreSQL se levanta con `docker-compose.yml` y sus credenciales se leen desde `.env`.

In [1]:
from pathlib import Path
import json
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
RAW_DIR = Path('data/raw')
postgres_url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ.get('POSTGRES_HOST', 'localhost')}:{os.environ.get('POSTGRES_PORT', '5432')}/{os.environ['POSTGRES_DB']}"
)
engine = create_engine(postgres_url)
RAW_DIR.exists(), sorted(p.name for p in RAW_DIR.iterdir())

(True,
 ['olist_order_items_dataset.csv',
  'olist_orders_dataset.csv',
  'product_categories.json',
  'product_category_name_translation.csv'])

## 2. DataFrame 1 - PostgreSQL: order_items

Este DataFrame se consulta desde la tabla `order_items` cargada en PostgreSQL. La fuente original es el archivo CSV de Kaggle `olist_order_items_dataset.csv`, convertido previamente a tabla de base de datos.

In [2]:
df_order_items_pg = pd.read_sql('SELECT * FROM order_items', engine)
df_order_items_pg.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [3]:
pd.DataFrame({
    'columna': df_order_items_pg.columns,
    'tiene_nulos': df_order_items_pg.isna().any().values,
    'valores_faltantes': df_order_items_pg.isna().sum().values,
})

,columna,tiene_nulos,valores_faltantes
0,order_id,False,0
1,order_item_id,False,0
2,product_id,False,0
3,seller_id,False,0
4,shipping_limit_date,False,0
5,price,False,0
6,freight_value,False,0


In [4]:
df_order_items_pg[['price', 'freight_value']].agg(['mean', 'max', 'min']).round(2)

,price,freight_value
mean,120.65,19.99
max,6735.00,409.68
min,0.85,0.00


In [5]:
df_order_items_pg.groupby('seller_id')[['price', 'freight_value']].agg(['max', 'min']).head(10).round(2)

price        freight_value       
                                     max    min           max    min
seller_id                                                           
0015a82c2db000af6aaaf3ae2ecb0532  895.00  895.0         21.02  21.02
001cca7ae9ae17fb1caed9dfb1094831  199.00   69.9        114.62  14.72
001e6ad469a905060d959994f1b41e4f  250.00  250.0         17.94  17.94
002100f778ceb8431b7a1020ff7ab48f  129.90    9.9         34.15   4.91
003554e2dce176b5555353e4f3555ac8  120.00  120.0         19.38  19.38
004c9cd9d87a3c30c522c48c4fc07416  259.99   47.9        133.73   7.67
00720abe85ba0859807595bbf045a33b  132.00   13.5         38.60   6.10
00ab3eff1b5192e5f1a63bcecfee11c8   98.00   98.0         12.08  12.08
00d8b143d12632bad99c0ad66ad52825   86.00   86.0         51.10  51.10
00ee68308b45bc5e2660cd833c3f81cc  450.00   48.0         98.02   2.34

## 3. DataFrame 2 - CSV: orders

Este DataFrame se lee directamente desde el archivo CSV `olist_orders_dataset.csv`. Se parsean fechas para calcular dias de procesamiento entre compra y entrega al transportista.

In [6]:
date_cols = ['order_purchase_timestamp', 'order_delivered_carrier_date']
df_orders_csv = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv', parse_dates=date_cols)
df_orders_csv['processing_days'] = (
    df_orders_csv['order_delivered_carrier_date'] - df_orders_csv['order_purchase_timestamp']
).dt.total_seconds() / 86400
df_orders_csv.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,processing_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2.373924
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.742627
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,0.216100
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,3.758252
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,0.936053


In [7]:
pd.DataFrame({
    'columna': df_orders_csv.columns,
    'tiene_nulos': df_orders_csv.isna().any().values,
    'valores_faltantes': df_orders_csv.isna().sum().values,
})

,columna,tiene_nulos,valores_faltantes
0,order_id,False,0
1,customer_id,False,0
2,order_status,False,0
3,order_purchase_timestamp,False,0
4,order_approved_at,True,160
5,order_delivered_carrier_date,True,1783
6,order_delivered_customer_date,True,2965
7,order_estimated_delivery_date,False,0
8,processing_days,True,1783


In [8]:
df_orders_csv[['processing_days']].agg(['mean', 'max', 'min']).round(2)

,processing_days
mean,3.23
max,125.78
min,-171.21


In [9]:
df_orders_csv.groupby('order_status')[['processing_days']].agg(['max', 'min']).round(2)

processing_days        
                         max     min
order_status                        
approved                 NaN     NaN
canceled               40.42    0.37
created                  NaN     NaN
delivered             125.78 -171.21
invoiced                 NaN     NaN
processing               NaN     NaN
shipped                43.67   -0.05
unavailable              NaN     NaN

## 4. DataFrame 3 - JSON: product_categories

Este DataFrame se genera desde `product_categories.json`, archivo JSON creado a partir de la tabla de traduccion de categorias del dataset Kaggle. Se agregaron campos numericos de longitud de texto para poder analizar metricas.

In [10]:
df_categories_json = pd.read_json(RAW_DIR / 'product_categories.json')
df_categories_json.head()

,product_category_name,product_category_name_english,category_name_length,english_name_length,first_letter
0,beleza_saude,health_beauty,12,13,H
1,informatica_acessorios,computers_accessories,22,21,C
2,automotivo,auto,10,4,A
3,cama_mesa_banho,bed_bath_table,15,14,B
4,moveis_decoracao,furniture_decor,16,15,F


In [11]:
pd.DataFrame({
    'columna': df_categories_json.columns,
    'tiene_nulos': df_categories_json.isna().any().values,
    'valores_faltantes': df_categories_json.isna().sum().values,
})

,columna,tiene_nulos,valores_faltantes
0,product_category_name,False,0
1,product_category_name_english,False,0
2,category_name_length,False,0
3,english_name_length,False,0
4,first_letter,False,0


In [12]:
df_categories_json[['category_name_length', 'english_name_length']].agg(['mean', 'max', 'min']).round(2)

,category_name_length,english_name_length
mean,16.97,16.07
max,46.00,39.00
min,3.00,3.00


In [13]:
df_categories_json.groupby('first_letter')[['category_name_length', 'english_name_length']].agg(['max', 'min']).head(10).round(2)

category_name_length     english_name_length    
                              max min                 max min
first_letter                                                 
A                              25   5                  26   3
B                              22   5                  22   4
C                              34   3                  31   9
D                              15   7                  19   6
E                              11  11                  11  11
F                              30   6                  33   4
G                              18  18                  12  12
H                              21  12                  17  10
I                              29  29                  30  30
K                              46  46                  39  39

## 5. Observaciones generales

- El flujo integra fuentes heterogeneas: PostgreSQL, CSV y JSON.
- Los datos de ventas permiten medir precios, fletes, estados de orden y categorias de producto.
- La separacion entre descarga, carga y analisis facilita repetir el ETL sin rehacer el proyecto manualmente.

# Aplicacion Profesional de la Practica

## Carlos Diaz

En mi contexto profesional me interesa aplicar estos conocimientos en areas de ecommerce, operaciones digitales y automatizacion comercial. En estos entornos se generan datos de ventas, productos, clientes, pagos, conversaciones de WhatsApp, campanas publicitarias, costos de envio, estados de entrega y eventos de seguimiento. Estos datos normalmente viven en sistemas separados, por ejemplo plataformas de tienda, hojas de calculo, bases transaccionales, CRMs, APIs de publicidad y archivos descargados manualmente.

PostgreSQL, Docker y Python pueden organizar ese flujo de una manera mas profesional. Docker permite levantar una base de datos aislada y repetible, sin depender de configuraciones manuales de una computadora especifica. PostgreSQL sirve como repositorio estructurado para almacenar tablas limpias de pedidos, productos, clientes o pagos. Python permite automatizar la lectura de archivos CSV y JSON, transformar columnas, validar valores faltantes, calcular indicadores y cargar resultados hacia la base de datos.

Implementar un proceso ETL aportaria orden, trazabilidad y velocidad. En lugar de revisar reportes desconectados, la organizacion podria tener un flujo donde los datos se extraen desde sus fuentes, se transforman con reglas claras y se cargan en una base lista para analisis. Esto reduce errores humanos, mejora la consistencia de los reportes y permite repetir el proceso cada semana o cada dia.

Con la informacion integrada se podrian resolver decisiones concretas: identificar productos con mejor margen, detectar demoras logisticas, comparar desempeno por canal de venta, medir conversion real de campanas, encontrar pedidos con problemas y priorizar acciones comerciales. Tambien se podrian crear tableros de seguimiento para que gerencia vea ventas, costos y cumplimiento operativo con datos actualizados. En resumen, la practica conecta herramientas tecnicas con una necesidad real: convertir datos dispersos en informacion confiable para decidir mejor.